# Bronze - E-commerce Pedidos

## 1. Objetivo

Este notebook realiza a ingestão dos pedidos provenientes do SQL Server para a camada Bronze do Data Lake.

O processo tem como objetivos:

* Ler os pedidos da fonte de dados;
* Identificar cada versão de um pedido por meio de um hash;
* Classificar os registros como NOVO, ATUALIZACAO ou DUPLICADO;
* Persistir fisicamente na camada Bronze apenas versões ainda não processadas;
* Manter o histórico das versões processadas no Ingestion Control Log;
* Particionar os dados da Bronze pela data da última atualização do pedido;
* Preparar a estrutura para futuras atualizações constantes por meio de um pool.


## 2. Autenticação e configurações

### 2.1 Bibliotecas utilizadas

As bibliotecas utilizadas no processo são carregadas nesta etapa.


In [0]:
from pyspark.sql.functions import (
    col,
    lit,
    sha2,
    concat_ws,
    coalesce,
    when,
    current_timestamp
)

import pandas as pd

from deltalake import DeltaTable, write_deltalake

###2.2 Configuração dos caminhos e parâmetros

As credenciais de acesso são obtidas por meio do notebook de autenticação.

Os dados da camada Bronze serão armazenados fisicamente no Data Lake, enquanto o histórico de ingestão será armazenado em uma estrutura de metadados separada.

In [0]:
%run ./00_autenticação

In [0]:
storage_account = "internshipdatalake"
lake_container = "squad1"

bronze_path = (
    f"abfss://{lake_container}@"
    f"{storage_account}.dfs.core.windows.net/"
    f"bronze/ecommerce_pedidos"
)

control_path = (
    f"abfss://{lake_container}@"
    f"{storage_account}.dfs.core.windows.net/"
    f"metadata/ingestion_control_log"
)

storage_options = {
    "AZURE_STORAGE_ACCOUNT_NAME": storage_account,
    "AZURE_STORAGE_CLIENT_ID": client_id,
    "AZURE_STORAGE_TENANT_ID": tenant_id,
    "AZURE_STORAGE_CLIENT_SECRET": client_secret
}

## 3. Leitura dos dados de origem

### 3.1 Leitura da tabela de pedidos

Os pedidos são lidos diretamente da tabela `ecommerce_pedidos` do SQL Server utilizando a conexão JDBC configurada no notebook de autenticação.


In [0]:
df_origem = spark.read.jdbc(
    url=jdbc_url,
    table="ecommerce_pedidos",
    properties=connection_properties
)

### 3.2 Validação dos dados carregados

Nesta etapa são verificadas a quantidade de registros e a estrutura dos dados provenientes da fonte.


In [0]:
print("Quantidade de registros:", df_origem.count())

df_origem.printSchema()

display(df_origem)

##4. Preparação dos dados
###4.1 Geração do hash do registro

O hash representa o conteúdo da versão de cada pedido.

A combinação entre id_pedido e hash_registro será utilizada para identificar se uma determinada versão já foi processada.

Dessa forma, alterações nos dados de um pedido resultarão em um novo hash e poderão ser identificadas como uma nova versão.

In [0]:
df_origem = df_origem.withColumn(
    "hash_registro",
    sha2(
        concat_ws(
            "||",
            coalesce(col("id_pedido").cast("string"), lit("")),
            coalesce(col("id_cliente").cast("string"), lit("")),
            coalesce(col("id_endereco_entrega").cast("string"), lit("")),
            coalesce(col("dt_pedido").cast("string"), lit("")),
            coalesce(col("status_pedido"), lit("")),
            coalesce(col("valor_total").cast("string"), lit("")),
            coalesce(col("valor_frete").cast("string"), lit("")),
            coalesce(col("metodo_pagamento"), lit("")),
            coalesce(
                col("dt_ultima_atualizacao_status").cast("string"),
                lit("")
            )
        ),
        256
    )
)

### 4.2 Criação da coluna de particionamento

A camada Bronze será particionada pela data da última atualização do status do pedido.

A coluna será utilizada para organizar fisicamente os arquivos no Data Lake, facilitando a leitura dos dados por períodos.


In [0]:
df_origem = df_origem.withColumn(
    "data_ultima_atualizacao",
    col("dt_ultima_atualizacao_status").cast("date")
)

##5. Consulta do histórico de ingestão
###5.1 Leitura do histórico de ingestão

O Ingestion Control Log contém as versões de pedidos que já foram processadas anteriormente.

A consulta ao histórico permite identificar se cada registro recebido é uma versão nova ou uma versão que já foi processada.

In [0]:
dt_control = DeltaTable(
    control_path,
    storage_options=storage_options
)

df_historico = dt_control.to_pandas()

print(
    "Registros no histórico:",
    len(df_historico)
)

### 5.2 Preparação das chaves históricas

Para a comparação são utilizadas duas referências:

* `id_pedido`: identifica se o pedido já é conhecido;
* `id_pedido + hash_registro`: identifica se aquela versão específica do pedido já foi processada.


In [0]:
df_historico_chaves = (
    df_historico[
        ["id_pedido", "hash_registro"]
    ]
    .drop_duplicates()
)

df_historico_ids = (
    df_historico[
        ["id_pedido"]
    ]
    .drop_duplicates()
)

In [0]:
df_historico_chaves_spark = spark.createDataFrame(
    df_historico_chaves
)

df_historico_ids_spark = spark.createDataFrame(
    df_historico_ids
)

## 6. Classificação dos registros

### 6.1 Preparação das referências históricas

São criadas marcações para indicar quando um pedido ou uma versão específica já foi encontrada no histórico.


In [0]:
df_historico_chaves_spark = (
    df_historico_chaves_spark
    .withColumn(
        "encontrado_historico",
        lit(1)
    )
)

df_historico_ids_spark = (
    df_historico_ids_spark
    .withColumn(
        "id_existente_historico",
        lit(1)
    )
)

### 6.2 Classificação das ocorrências

Os registros são classificados de acordo com sua existência no histórico:

* **NOVO:** o `id_pedido` ainda não existe no histórico;
* **DUPLICADO:** o `id_pedido` existe e a combinação `id_pedido + hash_registro` já existe;
* **ATUALIZACAO:** o `id_pedido` existe, mas a combinação `id_pedido + hash_registro` ainda não existe.


In [0]:
df_classificacao = df_origem.join(
    df_historico_chaves_spark,
    on=["id_pedido", "hash_registro"],
    how="left"
)

df_classificacao = df_classificacao.join(
    df_historico_ids_spark,
    on="id_pedido",
    how="left"
)

df_classificacao = df_classificacao.withColumn(
    "tipo_ocorrencia",
    when(
        col("id_existente_historico").isNull(),
        "NOVO"
    ).when(
        col("encontrado_historico").isNotNull(),
        "DUPLICADO"
    ).otherwise(
        "ATUALIZACAO"
    )
)

### 6.3 Validação da classificação

Antes da persistência dos dados, é apresentada a quantidade de registros identificados em cada tipo de ocorrência.


In [0]:
display(
    df_classificacao
    .groupBy("tipo_ocorrencia")
    .count()
    .orderBy("tipo_ocorrencia")
)

## 7. Persistência no Bronze

### 7.1 Seleção dos registros

Somente registros classificados como `NOVO` ou `ATUALIZACAO` serão persistidos na camada Bronze.

Registros classificados como `DUPLICADO` não serão persistidos novamente.


In [0]:
df_para_bronze = df_classificacao.filter(
    col("tipo_ocorrencia").isin(
        "NOVO",
        "ATUALIZACAO"
    )
)

df_bronze_final = df_para_bronze.drop(
    "tipo_ocorrencia",
    "encontrado_historico",
    "id_existente_historico"
)

print(
    "Registros para persistir na Bronze:",
    df_bronze_final.count()
)

### 7.2 Persistência física dos dados

Os dados classificados como novas versões são persistidos fisicamente no caminho da camada Bronze.

A persistência utiliza Delta Lake e particiona os dados pela `data_ultima_atualizacao`.

Registros classificados como `DUPLICADO` não são gravados novamente.


In [0]:
if df_bronze_final.limit(1).count() > 0:

    df_bronze_pandas = df_bronze_final.toPandas()

    write_deltalake(
        bronze_path,
        df_bronze_pandas,
        mode="append",
        partition_by=["data_ultima_atualizacao"],
        storage_options=storage_options
    )

    print("Novas versões persistidas na Bronze.")

else:

    print("Nenhuma nova versão para persistir na Bronze.")

## 8. Atualização do Ingestion Control Log

### 8.1 Seleção das novas versões

O Ingestion Control Log deve registrar somente versões que ainda não existem no histórico.

Portanto, registros classificados como `DUPLICADO` não são inseridos novamente no controle.


In [0]:
df_novas_versoes = df_classificacao.filter(
    col("tipo_ocorrencia").isin(
        "NOVO",
        "ATUALIZACAO"
    )
)

print(
    "Novas versões para registrar:",
    df_novas_versoes.count()
)

### 8.2 Preparação dos dados de controle

São selecionadas as informações necessárias para controlar as versões já processadas e registrada a data em que a versão foi inserida no histórico.


In [0]:
df_novas_versoes = df_novas_versoes.select(
    "id_pedido",
    "hash_registro",
    "dt_ultima_atualizacao_status",
    "tipo_ocorrencia"
)

df_novas_versoes = df_novas_versoes.withColumn(
    "data_ingestao",
    current_timestamp()
)

print(
    "Colunas do controle:",
    df_novas_versoes.columns
)

### 8.3 Persistência do controle de ingestão

Somente quando existirem novas versões, o histórico de ingestão será atualizado.

Caso não existam novas versões, nenhuma informação será adicionada ao Control Log.


In [0]:
df_novas_versoes_pandas = df_novas_versoes.toPandas()

if len(df_novas_versoes_pandas) > 0:

    write_deltalake(
        control_path,
        df_novas_versoes_pandas,
        mode="append",
        storage_options=storage_options
    )

    print("Control Log atualizado.")

else:

    print("Nenhuma nova versão para registrar no Control Log.")

## 9. Validações

### 9.1 Validação da camada Bronze

Nesta etapa é validada a existência da tabela Delta física e a quantidade de registros armazenados na camada Bronze.


In [0]:
dt_bronze = DeltaTable(
    bronze_path,
    storage_options=storage_options
)

df_bronze_validacao = dt_bronze.to_pandas()

print(
    "Versão do Delta Bronze:",
    dt_bronze.version()
)

print(
    "Registros na Bronze:",
    len(df_bronze_validacao)
)

### 9.2 Validação do controle de ingestão

O histórico de ingestão é validado considerando tanto a quantidade de registros armazenados quanto a quantidade de versões únicas.


In [0]:
dt_control_validacao = DeltaTable(
    control_path,
    storage_options=storage_options
)

df_control_validacao = dt_control_validacao.to_pandas()

print(
    "Registros no Control Log:",
    len(df_control_validacao)
)

print(
    "Pedidos distintos:",
    df_control_validacao["id_pedido"].nunique()
)

print(
    "Versões distintas:",
    df_control_validacao[
        ["id_pedido", "hash_registro"]
    ].drop_duplicates().shape[0]
)

## 10. Ingestão de dados em tempo real

A ingestão em tempo real tem como objetivo identificar novos arquivos disponibilizados no Data Lake e incorporá-los à camada Bronze de forma incremental.

O processo inicia-se pelo monitoramento da área destinada aos dados em tempo real. Quando um novo arquivo é identificado, seus registros são lidos, classificados e preparados para persistência na camada Bronze.

Os registros processados também são registrados no Ingestion Control Log, permitindo o controle das versões ingeridas e evitando o processamento duplicado de registros.
